In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PATH DEFINING

In [ ]:
#defining paths
import os

BASE_DIR = "/content/drive/MyDrive/multimodal_project"

VIDEO_DIR = os.path.join(
    BASE_DIR,
    "dataset",
    "subset_videos"
)

FRAME_DIR = os.path.join(
    BASE_DIR,
    "frames"
)

AUDIO_DIR = os.path.join(
    BASE_DIR,
    "audio"
)

IMAGE_EMB_DIR = os.path.join(
    BASE_DIR,
    "image_embeddings"
)

AUDIO_EMB_DIR = os.path.join(
    BASE_DIR,
    "audio_embeddings"
)

VIDEO_EMB_DIR = os.path.join(
    BASE_DIR,
    "video_embeddings"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs"
)

CHECKPOINT_DIR=os.path.join(
    BASE_DIR,
    "checkpoints"
)
print("Video Count:",
      len(os.listdir(VIDEO_DIR)))

LOADING EMBEDDINGS OF MIDDLE FRMAE , AUDIO AND COMPLETE VIDEO

In [ ]:
#load all image/audio and video embeddings
import numpy as np
import pandas as pd
import os

common_ids = pd.read_csv(
    f"{OUTPUT_DIR}/common_samples.csv"
)["video_id"].tolist()

image_embeddings = []
audio_embeddings = []
video_embeddings = []

for vid in common_ids:

    image_embeddings.append(
        np.load(
            os.path.join(
                IMAGE_EMB_DIR,
                vid + ".npy"
            )
        ).squeeze()
    )

    audio_embeddings.append(
        np.load(
            os.path.join(
                AUDIO_EMB_DIR,
                vid + ".npy"
            )
        ).squeeze()
    )

    video_embeddings.append(
        np.load(
            os.path.join(
                VIDEO_EMB_DIR,
                vid + ".npy"
            )
        ).squeeze()
    )

image_embeddings = np.array(image_embeddings)
audio_embeddings = np.array(audio_embeddings)
video_embeddings = np.array(video_embeddings)

DATASET PREPARATION

In [ ]:
#train/test/val split
from sklearn.model_selection import train_test_split

img_train,img_temp,\
aud_train,aud_temp,\
vid_train,vid_temp = train_test_split(
    image_embeddings,
    audio_embeddings,
    video_embeddings,
    test_size=0.30,
    random_state=42
)

img_val,img_test,\
aud_val,aud_test,\
vid_val,vid_test = train_test_split(
    img_temp,
    aud_temp,
    vid_temp,
    test_size=0.50,
    random_state=42
)

In [ ]:
#dataset
import torch
from torch.utils.data import Dataset

class FusionDataset(Dataset):

    def __init__(
        self,
        image,
        audio,
        video
    ):

        self.image = torch.tensor(
            image,
            dtype=torch.float32
        )

        self.audio = torch.tensor(
            audio,
            dtype=torch.float32
        )

        self.video = torch.tensor(
            video,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.image)

    def __getitem__(self, idx):

        return (
            self.image[idx],
            self.audio[idx],
            self.video[idx]
        )

In [ ]:
#dataloader
from torch.utils.data import DataLoader

train_loader = DataLoader(
    FusionDataset(
        img_train,
        aud_train,
        vid_train
    ),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    FusionDataset(
        img_val,
        aud_val,
        vid_val
    ),
    batch_size=32
)

test_loader = DataLoader(
    FusionDataset(
        img_test,
        aud_test,
        vid_test
    ),
    batch_size=32
)

TRANSFORMER MODEL WITH INFONCE LOSS

In [ ]:
#transformer fusion model
import torch
import torch.nn as nn

class TransformerFusion(nn.Module):

    def __init__(self):

        super().__init__()

        self.image_proj = nn.Linear(
            512,
            256
        )

        self.audio_proj = nn.Linear(
            128,
            256
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=8,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.projector = nn.Sequential(

            nn.Linear(
                256,
                512
            ),

            nn.ReLU(),

            nn.Linear(
                512,
                768
            )
        )

    def forward(
        self,
        image,
        audio
    ):

        img = self.image_proj(
            image
        )

        aud = self.audio_proj(
            audio
        )

        tokens = torch.stack(
            [img, aud],
            dim=1
        )

        fused = self.transformer(
            tokens
        )

        fused = fused.mean(
            dim=1
        )

        return self.projector(
            fused
        )

In [ ]:
#infonce loss
import torch.nn.functional as F

def info_nce_loss(
    pred,
    target,
    temperature=0.07
):

    pred = F.normalize(
        pred,
        dim=1
    )

    target = F.normalize(
        target,
        dim=1
    )

    logits = (
        pred @ target.T
    ) / temperature

    labels = torch.arange(
        pred.size(0)
    ).to(pred.device)

    return F.cross_entropy(
        logits,
        labels
    )

TRAINING MODEL

In [ ]:
#train
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = TransformerFusion().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

EPOCHS = 20

best_val = 999

In [ ]:
train_losses = []
val_losses = []

for epoch in range(EPOCHS):

    model.train()

    running = 0

    for img,aud,vid in train_loader:

        img = img.to(device)
        aud = aud.to(device)
        vid = vid.to(device)

        optimizer.zero_grad()

        pred = model(
            img,
            aud
        )

        loss = info_nce_loss(
            pred,
            vid
        )

        loss.backward()

        optimizer.step()

        running += loss.item()

    train_loss = running / len(train_loader)

    train_losses.append(
        train_loss
    )

    model.eval()

    val_running = 0

    with torch.no_grad():

        for img,aud,vid in val_loader:

            img = img.to(device)
            aud = aud.to(device)
            vid = vid.to(device)

            pred = model(
                img,
                aud
            )

            loss = info_nce_loss(
                pred,
                vid
            )

            val_running += loss.item()

    val_loss = val_running / len(val_loader)

    val_losses.append(
        val_loss
    )

    print(
        f"Epoch {epoch+1}"
        f" | Train {train_loss:.4f}"
        f" | Val {val_loss:.4f}"
    )

    if val_loss < best_val:

        best_val = val_loss

        torch.save(
            model.state_dict(),
            f"{CHECKPOINT_DIR}/transformer_contrastive.pth"
        )

GENERATING TEST SET PREDICTIONS AND EVALUATION

In [ ]:
#test
all_preds = []
all_targets = []

model.eval()

with torch.no_grad():

    for img,aud,vid in test_loader:

        img = img.to(device)
        aud = aud.to(device)

        pred = model(
            img,
            aud
        )

        all_preds.append(
            pred.cpu().numpy()
        )

        all_targets.append(
            vid.numpy()
        )

all_preds = np.vstack(
    all_preds
)

all_targets = np.vstack(
    all_targets
)

In [ ]:
#cosine simlarity
from sklearn.preprocessing import normalize

preds = normalize(
    all_preds
)

targets = normalize(
    all_targets
)

diag_sim = np.mean(
    np.sum(
        preds * targets,
        axis=1
    )
)

print(
    "Cosine Similarity:",
    diag_sim
)

In [ ]:
#RETRIEVAL METRICS
similarity_matrix = preds @ targets.T

ranks = []

for i in range(len(preds)):

    sims = similarity_matrix[i]

    sorted_idx = np.argsort(
        sims
    )[::-1]

    rank = np.where(
        sorted_idx == i
    )[0][0] + 1

    ranks.append(rank)

r1 = np.mean(
    np.array(ranks) <= 1
)

r5 = np.mean(
    np.array(ranks) <= 5
)

r10 = np.mean(
    np.array(ranks) <= 10
)

medr = np.median(
    ranks
)

print(
    f"Recall@1 : {r1*100:.2f}%"
)

print(
    f"Recall@5 : {r5*100:.2f}%"
)

print(
    f"Recall@10 : {r10*100:.2f}%"
)

print(
    f"Median Rank : {medr}"
)

In [ ]:
import numpy as np

mse = np.mean(
    (all_preds - all_targets) ** 2
)

print("Test MSE:", mse)

TRANSFORMER WITH MSE+COSINE LOSS





In [ ]:
#transforer model
class TransformerFusion(nn.Module):

    def __init__(self):

        super().__init__()

        self.image_proj = nn.Linear(
            512,
            256
        )

        self.audio_proj = nn.Linear(
            128,
            256
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=8,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.output = nn.Sequential(

            nn.Linear(256,512),
            nn.ReLU(),
            nn.Linear(512,768)
        )

    def forward(self,image,audio):

        image_token = self.image_proj(image)

        audio_token = self.audio_proj(audio)

        tokens = torch.stack(
            [image_token,audio_token],
            dim=1
        )

        fused = self.transformer(tokens)

        fused = fused.mean(dim=1)

        return self.output(fused)

In [ ]:
#loss
import torch.nn.functional as F
import torch.nn as nn

mse_loss = nn.MSELoss()

def regression_loss(
    pred,
    target
):

    mse = mse_loss(
        pred,
        target
    )

    cosine = 1 - F.cosine_similarity(
        pred,
        target,
        dim=1
    ).mean()

    return mse + cosine

In [ ]:
#intialising model
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = TransformerFusion().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

EPOCHS = 20

In [ ]:
#training
best_val = float("inf")

train_losses = []
val_losses = []

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0

    for img,aud,vid in train_loader:

        img = img.to(device)
        aud = aud.to(device)
        vid = vid.to(device)

        optimizer.zero_grad()

        pred = model(
            img,
            aud
        )

        loss = regression_loss(
            pred,
            vid
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for img,aud,vid in val_loader:

            img = img.to(device)
            aud = aud.to(device)
            vid = vid.to(device)

            pred = model(
                img,
                aud
            )

            loss = regression_loss(
                pred,
                vid
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch+1}"
        f" | Train {train_loss:.4f}"
        f" | Val {val_loss:.4f}"
    )

    if val_loss < best_val:

        best_val = val_loss

        torch.save(
            model.state_dict(),
            f"{CHECKPOINT_DIR}/transformer_regression.pth"
        )

TEST SET PREDICTIONS AND EVALUATION

In [ ]:
#test
all_preds = []
all_targets = []

model.eval()

with torch.no_grad():

    for img,aud,vid in test_loader:

        img = img.to(device)
        aud = aud.to(device)

        pred = model(
            img,
            aud
        )

        all_preds.append(
            pred.cpu().numpy()
        )

        all_targets.append(
            vid.numpy()
        )

all_preds = np.vstack(
    all_preds
)

all_targets = np.vstack(
    all_targets
)

In [ ]:
#cosine simlarity
from sklearn.preprocessing import normalize

preds = normalize(
    all_preds
)

targets = normalize(
    all_targets
)

diag_sim = np.mean(
    np.sum(
        preds * targets,
        axis=1
    )
)

print(
    "Cosine Similarity:",
    diag_sim
)

In [ ]:
#RETRIEVAL METRICS
similarity_matrix = preds @ targets.T

ranks = []

for i in range(len(preds)):

    sims = similarity_matrix[i]

    sorted_idx = np.argsort(
        sims
    )[::-1]

    rank = np.where(
        sorted_idx == i
    )[0][0] + 1

    ranks.append(rank)

r1 = np.mean(
    np.array(ranks) <= 1
)

r5 = np.mean(
    np.array(ranks) <= 5
)

r10 = np.mean(
    np.array(ranks) <= 10
)

medr = np.median(
    ranks
)

print(
    f"Recall@1 : {r1*100:.2f}%"
)

print(
    f"Recall@5 : {r5*100:.2f}%"
)

print(
    f"Recall@10 : {r10*100:.2f}%"
)

print(
    f"Median Rank : {medr}"
)

In [ ]:
import numpy as np

mse = np.mean(
    (all_preds - all_targets) ** 2
)

print("Test MSE:", mse)

In [ ]:
print("Diagonal Mean:")

diag = np.mean(
    np.diag(
        similarity_matrix
    )
)

print(diag)

print("Overall Mean:")

print(
    similarity_matrix.mean()
)